# Porownanie modelu B0 (stary) vs B4 (nowy)

Folder `malignant` zawiera zdjecia z **potwierdzona diagnoza malignant** -- czyli
znamy prawdziwa etykiete dla kazdego zdjecia (wszystkie = malignant).

Dzieki temu mozemy bezposrednio policzyc **recall** obu modeli na tym samym zbiorze:
ile z prawdziwie zlosliwych zmian kazdy model faktycznie wykryl.

Jesli wczytanie modelu B0 rzuci blad `size mismatch` w `load_state_dict` -- oznacza to,
ze architektura (`B0_ARCH`) albo rozmiar wejscia (`B0_IMG_SIZE`) w konfiguracji ponizej
nie zgadza sie z tym, na czym faktycznie trenowales B0. Popraw te dwie wartosci i uruchom ponownie.


In [1]:
import torch
import torch.nn as nn
from torchvision import transforms
from PIL import Image
from pathlib import Path
from torchvision import models

import timm

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Urzadzenie:", device)


Urzadzenie: cpu


## Konfiguracja -- dostosuj sciezki i parametry B0

In [2]:
MALIGNANT_DIR = Path(r"C:\Users\Akadia Sobanska\Documents\AI_Data_Scientist\ML_picture_recognition\ML_picture_recognition_V1\melanoma_classification_2\malignant")

B0_PATH = Path(r"C:\Users\Akadia Sobanska\Documents\AI_Data_Scientist\ML_picture_recognition\ML_picture_recognition_V1\melanoma_classification_2\model_b0\melanoma_model.pth")

B0_IMG_SIZE = 224          # zmien jesli trenowales na innym rozmiarze
B0_ARCH = "efficientnet_b0"

B4_PATH = Path(r"C:\Users\Akadia Sobanska\Documents\AI_Data_Scientist\ML_picture_recognition\ML_picture_recognition_V1\melanoma_classification_2\model_nowy_b4\efficientnet_b4_melanoma_final.pt")
B4_IMG_SIZE = 380
B4_ARCH = "efficientnet_b4"

THRESHOLD = 0.5  # prog decyzyjny: prawdopodobienstwo >= THRESHOLD -> malignant

for p in [MALIGNANT_DIR, B0_PATH, B4_PATH]:
    print(p, "->", "OK" if p.exists() else "BRAK -- sprawdz sciezke")


C:\Users\Akadia Sobanska\Documents\AI_Data_Scientist\ML_picture_recognition\ML_picture_recognition_V1\melanoma_classification_2\malignant -> OK
C:\Users\Akadia Sobanska\Documents\AI_Data_Scientist\ML_picture_recognition\ML_picture_recognition_V1\melanoma_classification_2\model_b0\melanoma_model.pth -> OK
C:\Users\Akadia Sobanska\Documents\AI_Data_Scientist\ML_picture_recognition\ML_picture_recognition_V1\melanoma_classification_2\model_nowy_b4\efficientnet_b4_melanoma_final.pt -> OK


## Funkcje pomocnicze

- `build_transform` -- przygotowuje obraz (resize + normalizacja) pod dany rozmiar wejscia
- `load_model` -- wczytuje architekture i wagi z pliku .pt/.pth
- `predict_folder` -- puszcza wszystkie zdjecia z folderu przez model, zwraca prawdopodobienstwa


In [3]:
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]


def build_transform(img_size):
    return transforms.Compose([
        transforms.Resize((img_size, img_size)),
        transforms.ToTensor(),
        transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
    ])


def load_model(arch, weights_path, device, use_torchvision=False, num_classes=1):
    if use_torchvision:
        model = models.efficientnet_b0(weights=None)
        model.classifier[1] = nn.Linear(model.classifier[1].in_features, num_classes)
    else:
        model = timm.create_model(arch, pretrained=False, num_classes=num_classes)

    state = torch.load(weights_path, map_location=device)
    if isinstance(state, dict) and "model_state_dict" in state:
        state = state["model_state_dict"]

    model.load_state_dict(state)
    model.to(device)
    model.eval()
    return model


def predict_folder(model, transform, folder, device, binary_output=True, malignant_idx=1):
    results = []
    image_paths = sorted(folder.glob("*.jpg")) + sorted(folder.glob("*.png"))
    with torch.no_grad():
        for path in image_paths:
            img = Image.open(path).convert("RGB")
            x = transform(img).unsqueeze(0).to(device)
            output = model(x)
            if binary_output:
                prob = torch.sigmoid(output).item()
            else:
                probs = torch.softmax(output, dim=1)
                prob = probs[0, malignant_idx].item()
            results.append((path.name, prob))
    return results


def summarize(name, results, threshold):
    total = len(results)
    correct = sum(1 for _, p in results if p >= threshold)
    recall = correct / total if total else 0.0
    print(f"\n{name}:")
    print(f"  Zdjec ogolem (wszystkie prawdziwie malignant): {total}")
    print(f"  Poprawnie wykryte jako malignant:               {correct}")
    print(f"  Przeoczone (falszywy negatyw):                  {total - correct}")
    print(f"  Recall:                                          {recall:.4f} ({recall*100:.1f}%)")
    return recall

print("Funkcje gotowe.")


Funkcje gotowe.


## Wczytanie modeli

Jesli B0 rzuci blad przy `load_state_dict` -- popraw `B0_ARCH`/`B0_IMG_SIZE` w komorce konfiguracji
powyzej i uruchom od nowa od tej komorki.


In [4]:
print("Wczytywanie modelu B0...")
b0_model = load_model(B0_ARCH, B0_PATH, device, use_torchvision=True, num_classes=2)
b0_transform = build_transform(B0_IMG_SIZE)

print("Wczytywanie modelu B4...")
b4_model = load_model(B4_ARCH, B4_PATH, device)
b4_transform = build_transform(B4_IMG_SIZE)

print("Oba modele wczytane poprawnie.")


Wczytywanie modelu B0...
Wczytywanie modelu B4...
Oba modele wczytane poprawnie.


## Inferencja na folderze malignant

Puszczamy wszystkie zdjecia przez oba modele. Moze to chwile potrwac zaleznie od liczby zdjec
i czy masz GPU.


In [8]:
print(f"Uruchamiam inferencje na folderze: {MALIGNANT_DIR}")

b0_results = predict_folder(b0_model, b0_transform, MALIGNANT_DIR, device, binary_output=False, malignant_idx=1)
b4_results = predict_folder(b4_model, b4_transform, MALIGNANT_DIR, device)

print(f"Przetworzono {len(b0_results)} zdjec (B0), {len(b4_results)} zdjec (B4)")


Uruchamiam inferencje na folderze: C:\Users\Akadia Sobanska\Documents\AI_Data_Scientist\ML_picture_recognition\ML_picture_recognition_V1\melanoma_classification_2\malignant
Przetworzono 505 zdjec (B0), 505 zdjec (B4)


## Wyniki -- recall obu modeli

In [9]:
b0_recall = summarize("MODEL B0 (stary)", b0_results, THRESHOLD)
b4_recall = summarize("MODEL B4 (nowy)", b4_results, THRESHOLD)

print(f"\nRoznica recall (B4 - B0): {(b4_recall - b0_recall)*100:+.1f} punktow procentowych")



MODEL B0 (stary):
  Zdjec ogolem (wszystkie prawdziwie malignant): 505
  Poprawnie wykryte jako malignant:               426
  Przeoczone (falszywy negatyw):                  79
  Recall:                                          0.8436 (84.4%)

MODEL B4 (nowy):
  Zdjec ogolem (wszystkie prawdziwie malignant): 505
  Poprawnie wykryte jako malignant:               474
  Przeoczone (falszywy negatyw):                  31
  Recall:                                          0.9386 (93.9%)

Roznica recall (B4 - B0): +9.5 punktow procentowych


## Szczegoly per obraz -- gdzie modele sie roznia

Pokazuje tylko przypadki, gdzie B0 i B4 daja rozna decyzje (jeden mowi malignant, drugi benign) --
to najciekawsze przypadki do recznego przejrzenia.


In [10]:
b0_dict = dict(b0_results)
b4_dict = dict(b4_results)

diffs = []
for name in b0_dict:
    b0_prob = b0_dict[name]
    b4_prob = b4_dict.get(name, None)
    if b4_prob is None:
        continue
    b0_pred = "malignant" if b0_prob >= THRESHOLD else "benign"
    b4_pred = "malignant" if b4_prob >= THRESHOLD else "benign"
    if b0_pred != b4_pred:
        diffs.append((name, b0_prob, b4_prob, b0_pred, b4_pred))

print(f"Liczba zdjec gdzie modele sie nie zgadzaja: {len(diffs)}\n")
for name, b0_prob, b4_prob, b0_pred, b4_pred in diffs:
    print(f"  {name}: B0={b0_pred} ({b0_prob:.2%})  vs  B4={b4_pred} ({b4_prob:.2%})")


Liczba zdjec gdzie modele sie nie zgadzaja: 54

  melanoma_10106.jpg: B0=benign (27.75%)  vs  B4=malignant (95.09%)
  melanoma_10107.jpg: B0=benign (2.57%)  vs  B4=malignant (75.67%)
  melanoma_10108.jpg: B0=benign (5.17%)  vs  B4=malignant (84.02%)
  melanoma_10114.jpg: B0=benign (20.93%)  vs  B4=malignant (59.72%)
  melanoma_10115.jpg: B0=benign (30.44%)  vs  B4=malignant (88.27%)
  melanoma_10117.jpg: B0=benign (17.21%)  vs  B4=malignant (73.24%)
  melanoma_10118.jpg: B0=benign (31.50%)  vs  B4=malignant (60.03%)
  melanoma_10119.jpg: B0=benign (6.33%)  vs  B4=malignant (96.34%)
  melanoma_10120.jpg: B0=benign (2.28%)  vs  B4=malignant (63.06%)
  melanoma_10123.jpg: B0=benign (0.97%)  vs  B4=malignant (81.54%)
  melanoma_10124.jpg: B0=benign (22.83%)  vs  B4=malignant (83.76%)
  melanoma_10127.jpg: B0=benign (45.63%)  vs  B4=malignant (93.70%)
  melanoma_10128.jpg: B0=benign (30.72%)  vs  B4=malignant (80.20%)
  melanoma_10131.jpg: B0=benign (5.08%)  vs  B4=malignant (94.10%)
  mela

## Podsumowanie

Uzupelnij po uruchomieniu:
- Recall B0: 0,44
- Recall B4: 0,65
- Wniosek: czy nowy model B4 rzeczywiscie poprawia wykrywanie malignant wzgledem B0?
